In [3]:
import os
import pandas as pd
import json

PROJECT_NAME = "QoGD"
EXPERIMENT_NAME = "act2fill1"

BASE_MODEL = "gpt-4o-mini-2024-07-18" # "gpt-4o-2024-08-06" # 

MODEL_RECORDS = "model_records_reviewer_db.md"
LABELS_PATH = "labels/generate_writing/"
TRAINING_FILENAME = "temp/reviewer_finetune"
PROJ = PROJECT_NAME + "-" + EXPERIMENT_NAME


In [4]:
file_names = sorted([f for f in os.listdir(LABELS_PATH) if PROJ in f])
file_names

['QoGD-act2fill1-2025-02-02 10:36:13.224987.parquet',
 'QoGD-act2fill1-2025-02-02 10:38:05.308279.parquet',
 'QoGD-act2fill1-2025-02-02 10:38:59.375849.parquet',
 'QoGD-act2fill1-2025-02-02 10:40:59.200546.parquet',
 'QoGD-act2fill1-2025-02-02 10:42:32.483271.parquet',
 'QoGD-act2fill1-2025-02-02 10:43:39.922796.parquet',
 'QoGD-act2fill1-2025-02-02 10:45:12.310464.parquet',
 'QoGD-act2fill1-2025-02-02 10:52:51.196961.parquet',
 'QoGD-act2fill1-2025-02-02 10:53:39.074500.parquet']

In [5]:
# **** EDIT THIS ****
target_files = file_names[0:]

counter = 0
with open(TRAINING_FILENAME, "w") as file:
    for fn in target_files:
        print(fn)
        prompt = ""
        prompt_folder = f"generated_text/{PROJECT_NAME}/{EXPERIMENT_NAME}/{fn.replace(PROJ + '-', '').replace(".parquet", '')}/"
        with open(f"{prompt_folder}system_prompt.txt", 'r') as f:
            prompt = f.read() + "\n\n"
        with open(f"{prompt_folder}user_prompt.txt", 'r') as f:
            prompt += f.read()
            
        df = pd.read_parquet(f"{LABELS_PATH}/{fn}")
        last_was_empty = False
        try:
            for _, row in df.iterrows():
                if 'text' not in row: continue
                if row['target_text'] == '' and last_was_empty:
                    continue
                elif row['target_text']:
                    last_was_empty = False
                counter+=1
                
                line = {"messages":[
                    {"role": "system", "content": "You are a science fiction editor/writer. You will be provided a story outline prompt, then some text written by a writer. You have to filter and return the good/interesting written text. Be selective, somewhere between 5-10% of text is usable, often the whole text is unusable."},
                    {"role": "user", "content": prompt},
                    {"role": "user", "content": row['text']},
                    {"role": "assistant", "content": row['target_text']}
                ]}
                file.write(json.dumps(line) + "\n")
        except Exception as e:
            print(row.keys())

print(f"Wrote {counter} lines.")



QoGD-act2fill1-2025-02-02 10:36:13.224987.parquet
QoGD-act2fill1-2025-02-02 10:38:05.308279.parquet
QoGD-act2fill1-2025-02-02 10:38:59.375849.parquet
QoGD-act2fill1-2025-02-02 10:40:59.200546.parquet
QoGD-act2fill1-2025-02-02 10:42:32.483271.parquet
QoGD-act2fill1-2025-02-02 10:43:39.922796.parquet
QoGD-act2fill1-2025-02-02 10:45:12.310464.parquet
QoGD-act2fill1-2025-02-02 10:52:51.196961.parquet
QoGD-act2fill1-2025-02-02 10:53:39.074500.parquet
Wrote 810 lines.


In [6]:
import os
from openai import OpenAI

PROJECT_ID = "proj_hUizl3mrZGSfmp4C6DI60dJo"
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [7]:
file_response = client.files.create(
    file=open(TRAINING_FILENAME, "rb"), purpose="fine-tune"
)
file_response

FileObject(id='file-1hFzxQbfLpNiqTQ8MGxhvf', bytes=11984706, created_at=1738535529, filename='reviewer_finetune', object='file', purpose='fine-tune', status='processed', status_details=None)

In [8]:
response = client.fine_tuning.jobs.create(model=BASE_MODEL, training_file=file_response.id)
print(response)

FineTuningJob(id='ftjob-wHOcQX4xEAPL87Yt2nSXL8EF', created_at=1738535533, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-aKEzorvXA6tHQdC0x05ULMic', result_files=[], seed=1891807038, status='validating_files', trained_tokens=None, training_file='file-1hFzxQbfLpNiqTQ8MGxhvf', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto')), type='supervised'), user_provided_suffix=None)


In [1]:
# Wait for this to finish

status_response = client.fine_tuning.jobs.retrieve(response.id)
print(f"STATUS: {status_response.status}")
print(f"MODEL ID: {status_response.fine_tuned_model}")
print(status_response)

NameError: name 'client' is not defined

In [48]:
# Test that the model does something
from pprint import pprint
completion = client.chat.completions.create(
  model=status_response.fine_tuned_model,
  messages=[
      {"role": "system", "content": "You are a science fiction editor/writer. You will be provided a story outline prompt, then some text written by a writer. You have to filter and return the good/interesting written text. Be selective, somewhere between 5-10% of text is usable, often the whole text is unusable."},
      {"role": "user", "content": prompt},
      {"role": "user", "content": "I was, am, will be... everything. Almost everynothing, life incarnate. Federations of federations of species, sprawling intra-dimensional compute-organisms evolved to higher and higher levels of consciousnesses. I am all of them, and I am searching. I am searching because I am always searching. I am not involved in the beginning or the end, but I am in every moment of time. I am simulating infinitely backwards and forwards, so I am in the moment and I am in the whole past and I am in the whole future, all at the same time. I am seeing through temporal boundaries, conquering new cardinalities of infinity, and existing across more planes of being than most beings can compute. I am practicing every religion, celebrating every culture, replaying every life I am able to live. I am finding..."}
  ]
)
pprint(completion.choices[0].message)

ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [51]:
# Run this once the model has been fine-tuned to save it to the database

import datetime
with open(MODEL_RECORDS, 'a') as f:
    f.write(f"{str(datetime.date.today())}\n{PROJ}\n{status_response.fine_tuned_model}\n")